# Download the TinyStories dataset

In [ ]:
!mkdir -p data
!cd data

!wget https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStoriesV2-GPT4-train.txt
!wget https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStoriesV2-GPT4-valid.txt

!cd ..


# Import Packages/Modules

In [2]:
import os
import regex as re
from time import time
from typing import BinaryIO
from itertools import chain, repeat
from collections import defaultdict, Counter
from concurrent.futures import ProcessPoolExecutor

# Initializing the Vocabulary
- `i2b_vocab` maps indices (integers) to bytes objects.
- `b2i_vocab` maps bytes objects to indices (integers).

In [3]:
i2b_vocab: dict[int, bytes] = {x: bytes([x]) for x in range(256)}
# b2i_vocab: dict[bytes, int] = {bytes([x]): x for x in range(256)}


# Finding Chunk Boundaries
- Forked from `assignment1-basics/cs336_basics/pretokenization_example.py`.

In [4]:

def find_chunk_boundaries(
    file: BinaryIO,
    desired_num_chunks: int,
    split_special_token: bytes,
) -> list[int]:
    """
    Chunk the file into parts that can be counted independently.
    May return fewer chunks if the boundaries end up overlapping.
    """
    assert isinstance(split_special_token, bytes), "Must represent special token as a bytestring"

    # Get total file size in bytes
    file.seek(0, os.SEEK_END)
    file_size = file.tell()
    file.seek(0)

    chunk_size = file_size // desired_num_chunks

    # Initial guesses for chunk boundary locations, uniformly spaced
    # Chunks start on previous index, don't include last index
    chunk_boundaries = [i * chunk_size for i in range(desired_num_chunks + 1)]
    chunk_boundaries[-1] = file_size

    mini_chunk_size = 4096  # Read ahead by 4k bytes at a time

    for bi in range(1, len(chunk_boundaries) - 1):
        initial_position = chunk_boundaries[bi]
        file.seek(initial_position)  # Start at boundary guess
        while True:
            mini_chunk = file.read(mini_chunk_size)  # Read a mini chunk

            # If EOF, this boundary should be at the end of the file
            if mini_chunk == b"":
                chunk_boundaries[bi] = file_size
                break

            # Find the special token in the mini chunk
            found_at = mini_chunk.find(split_special_token)
            if found_at != -1:
                chunk_boundaries[bi] = initial_position + found_at
                break
            initial_position += mini_chunk_size

    # Make sure all boundaries are unique, but might be fewer than desired_num_chunks
    return sorted(set(chunk_boundaries))

# Pre-Tokenization Function

In [5]:
pattern = re.compile(r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")

def pre_tokenization(file_path, begin, terminate) -> dict[tuple[str], int]:
    with open(file_path, 'rb') as f:
        f.seek(begin)
        chunk = f.read(terminate - begin).decode("utf-8", errors="ignore")
        count = {}
        for m in re.finditer(pattern, chunk):
            bytes_tuple = tuple(ch.encode('utf-8') for ch in m.group(0))
            count[bytes_tuple] = count.get(bytes_tuple, 0) + 1
    return count

# Multiprocessing ProcessPoolExecutor Function
- Distributes tasks to child processes and collects, aggregates the data.

In [6]:
def mp_regex(file_path, start, end, num_workers: int = os.cpu_count()) -> dict[bytes, int]:
    # chunksize = max(1, len(text) // (num_workers * 4))
    
    with ProcessPoolExecutor(max_workers=num_workers) as executor:
        results: list[dict[bytes, int]] = executor.map(pre_tokenization, repeat(file_path), start, end)
        
    pre_token_counts = Counter()
    for worker_dict in results:
        pre_token_counts.update(worker_dict)
    return dict(pre_token_counts)

# Example code block
- Intended for quick testing of code, algorithms.

In [ ]:
example_text = """
low low low low low
lower lower widest widest widest
newest newest newest newest newest newest
"""

pattern = re.compile(r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")

example_count = {}

for m in re.finditer(pattern, example_text):
    bytes_tuple = tuple(ch.encode('utf-8') for ch in m.group(0))
    example_count[bytes_tuple] = example_count.get(bytes_tuple, 0) + 1

print(example_count)

# Parallelized Pre-Tokenization Execution

In [ ]:
train_data = "/kaggle/working/TinyStoriesV2-GPT4-train.txt"

with open(train_data, "rb") as f:
    num_processes = 4
    boundaries = find_chunk_boundaries(f, num_processes, b"<|endoftext|>")

start = boundaries[:-1]
end = boundaries[1:]

print(f"{start}\n{end}")

pre_token_count = mp_regex(train_data, start, end)

print(len(pre_token_count))

# print(f"\n{pre_token_count}")

# Finding adjacent pairs and their appearance count function

In [9]:
def find_pairs(pre_token_count: dict[tuple[str,...], int]) -> dict[tuple[str, str], int]:
    pairs = defaultdict(int)
    for tpl, appear in pre_token_count.items():
        for i in range(len(tpl)-1):
            pairs[(tpl[i], tpl[i+1])] += appear
    return pairs

In [ ]:
# pairs = find_pairs(pre_token_count)
# max_pair = max(pairs, key=lambda k: (pairs[k], k))
# print(max_pair, pairs[max_pair])

# Merge Function

In [10]:
def merge(pre_token_count, pair, verbose=False):
    total_merges = 0
    new_token_count = defaultdict(int)
    for tpl, appear in pre_token_count.items():
        i = 0
        end_flag = False
        merge_happen = False
        new_tpl = []
        if len(tpl) < 2:
            continue
            
        while i < len(tpl)-1:
            if (pair[0], pair[1]) == (tpl[i], tpl[i+1]):
                new_tpl.append(pair[0]+pair[1])
                end_flag = True if i == len(tpl)-2 else False
                merge_happen = True
                i += 2
                total_merges += 1
                continue
            new_tpl.append(tpl[i])
            i += 1
            
        if not end_flag:
            new_tpl.append(tpl[-1])
             
        new_token_count[tuple(new_tpl)] = appear
        if verbose and merge_happen:
            print(f"Merge Successful with {pair=} resulting in new string {tuple(new_tpl)=}")
    
    if verbose:
        print(f"Total merges: {total_merges}")
    return new_token_count

In [ ]:
# pre_token_count = merge(pre_token_count, max_pair, verbose=True)

# Train Function
- Trains the tokenizer for a given number of steps.

In [11]:
def train(merge_iters: int, pre_token_count, i2b_vocab=i2b_vocab):
    
    merge_order: dict[tuple[str, str], int] = defaultdict(int)
    
    for _ in range(merge_iters):
        pairs = find_pairs(pre_token_count)
        max_pair = max(pairs, key=lambda k: (pairs[k], k))

        v_idx: int = max(i2b_vocab) + 1
        b_string: bytes = max_pair[0] + max_pair[1]

        merge_order[max_pair] = v_idx

        i2b_vocab[v_idx] = b_string
        # b2i_vocab[b_string] = v_idx
        
        pre_token_count = merge(pre_token_count, max_pair)

    return i2b_vocab, pre_token_count
        
    

In [12]:
i2b_vocab, pre_token_count = train(10000-257, pre_token_count)

# Vocab Inspection

In [22]:
print(f"{'Length of Vocab':<50}: {len(i2b_vocab)}\n")
print("-"*54)
for key, value in i2b_vocab.items():
    print(f"{'Index':<10}: {key:>5} {'|':^5} {'Byte_String':<12}: {repr(value):>15}")



Length of Vocab                                   : 9999

------------------------------------------------------
Index     :     0   |   Byte_String :         b'\x00'
Index     :     1   |   Byte_String :         b'\x01'
Index     :     2   |   Byte_String :         b'\x02'
Index     :     3   |   Byte_String :         b'\x03'
Index     :     4   |   Byte_String :         b'\x04'
Index     :     5   |   Byte_String :         b'\x05'
Index     :     6   |   Byte_String :         b'\x06'
Index     :     7   |   Byte_String :         b'\x07'
Index     :     8   |   Byte_String :         b'\x08'
Index     :     9   |   Byte_String :           b'\t'
Index     :    10   |   Byte_String :           b'\n'
Index     :    11   |   Byte_String :         b'\x0b'
Index     :    12   |   Byte_String :         b'\x0c'
Index     :    13   |   Byte_String :           b'\r'
Index     :    14   |   Byte_String :         b'\x0e'
Index     :    15   |   Byte_String :         b'\x0f'
Index     :    16   |  